In [ ]:
import tensorflow as tf
print('TensorFlow version' + tf.__version__)
import json
import keras_hub
import os
import re
import numpy as np
import random
import nltk


with open("/kaggle/input/datasets/riccardoghilotti/ai-dataset/bert_config.json", "r") as jsonfile: 
    config = json.load(jsonfile)

bert_name = "bert_base_en_uncased"
MASK_RATE = 0.20
PREDICTIONS_PER_SEQ = 32
PRETRAINING_BATCH_SIZE = 32
SEQ_LENGTH = 512
VOCAB_SIZE = config["vocab_size"]

TensorFlow version2.19.0


## Prepare the dataset

In [ ]:
preprocessor = keras_hub.models.BertMaskedLMPreprocessor.from_preset(
    bert_name,
    mask_selection_rate=MASK_RATE, # Probability to mask a token
    mask_selection_length=PREDICTIONS_PER_SEQ, # Max number of masked tokens
)

In [ ]:
def preprocess(inputs, wrong_proba = 0.5):
    
    first_sentences = []
    second_sentences = []
    ground_truth = dict() 
    ground_truth["cls_output"] = []
    # Construct examples of consecutive sentences along with their corresponding ground truth (GT) 
    for i in range(len(inputs) - 1): # -1 -> Cannot build a couple using the last sentence (no consecutive sentences available)
        first_sent = inputs[i]
        
        r = random.random()
        if r > wrong_proba:
            # Wrong couple of consecutive sentences
            second_sent_index = random.randint(0, len(inputs) - 1) 
            while second_sent_index == (i + 1) or second_sent_index == i: # Avoid trivial cases
                second_sent_index = random.randint(0, len(inputs) - 1)
            second_sent = inputs[second_sent_index]
            ground_truth["cls_output"].append(0)
        else:
            # Correct couple of consecutive sentences
            second_sent = inputs[i + 1]
            ground_truth["cls_output"].append(1)
            
        second_sentences.append(second_sent)
        first_sentences.append(first_sent)


    ground_truth["cls_output"] = tf.convert_to_tensor(
        ground_truth["cls_output"],
        dtype=tf.int32
    )
    
    first_sentences = tf.constant(first_sentences)
    second_sentences = tf.constant(second_sentences)

    # Pre process the sentences using Bert Masked Preprocessor
    outputs = preprocessor((first_sentences, second_sentences))
    
    dictionary = outputs[0]
    # BERT input features
    features = {
        "token_ids": dictionary["token_ids"],
        "padding_mask": dictionary["padding_mask"],
        "segment_ids": dictionary["segment_ids"]
    }
    
    # Reconstruct a sequence-length representation by placing each ground-truth token
    # at its corresponding masked position; all non-masked positions remain zero
    token_pred_output_RAW = outputs[1] 
    masks_positions_RAW = dictionary["mask_positions"] 
    n_rows = token_pred_output_RAW.shape[0]
    
    token_pred_output = np.zeros((n_rows, SEQ_LENGTH))

    for j in range(n_rows):
        masks_position = masks_positions_RAW[j]
        for i, mask_position in enumerate(masks_position):
            if mask_position == 0:
                break
            token_pred_output[j, mask_position] = token_pred_output_RAW[j, i] 
    
    # Store the reconstructed ground-truth tensor for the masked token prediction task
    ground_truth["token_pred_output"] = tf.convert_to_tensor(token_pred_output, dtype=tf.uint16)
    return features, ground_truth

In [ ]:
nltk.download("punkt_tab", quiet=True)

# Compile regular expressions once to avoid recompilation overhead
RE_WIKI_HEADER = re.compile(r"^=+\s*.+?\s*=+\s*$", re.MULTILINE)
RE_EMAIL       = re.compile(r"\S+@\S+\.\S+")
RE_PHONE       = re.compile(r"[\+\d][\d\s\-\(\)]{6,}")
RE_URL         = re.compile(r"https?://\S+|www\.\S+")
RE_META        = re.compile(r"\([^)]{0,60}(IATA|updated|Metro|tram|bus|Train)[^)]*\)", re.IGNORECASE)
RE_HOURS       = re.compile(r"\b(?:M|Tu|W|Th|F|Sa|Su|Daily|Mon|Tue|Wed|Thu|Fri|Sat|Sun)[\w\s\-–:,]{3,30}\d{1,2}:\d{2}", re.IGNORECASE)
RE_PRICE       = re.compile(r"(?:€|£|\$|RUB|руб)\s*[\d,\.]+\+?|[\d,\.]+\s*(?:руб|RUB)")
RE_LIST_MARKER = re.compile(r"^\s*(\d+\s+|[•\-\*]\s+)")
RE_ADDRESS     = re.compile(r"\b(?:Address|Ulitsa|Prospekt|Pr\.|St\.|st\.|Str\.)\b", re.IGNORECASE)
RE_MULTI_SPACE = re.compile(r"\s{2,}")

# Abbreviations that should not trigger sentence splitting
RE_ABBREV = re.compile(
    r"\b(Mr|Mrs|Ms|Dr|Prof|St|Ave|vs|etc|approx|km|sq|ca|Jan|Feb|Mar|Apr|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\.",
    re.IGNORECASE,
)


def clean_text(raw: str) -> str:
    text = raw

    text = RE_WIKI_HEADER.sub(" ", text)   # Remove wiki-style section headers
    text = RE_URL.sub("", text)
    text = RE_EMAIL.sub("", text)
    text = RE_PHONE.sub("", text)
    text = RE_META.sub("", text)           # Remove metadata and transport annotations
    text = RE_HOURS.sub("", text)          # Remove opening hours and schedules
    text = RE_PRICE.sub("", text)

    text = text.replace("\n", " ")
    text = RE_LIST_MARKER.sub("", text)

    # Remove fragments that resemble standalone addresses
    fragments = text.split(".")
    fragments = [f for f in fragments if not RE_ADDRESS.search(f)]
    text = ". ".join(fragments)

    text = RE_MULTI_SPACE.sub(" ", text).strip()
    return text


def split_sentences(text: str) -> list[str]:
    return nltk.sent_tokenize(text)

def is_informative(sentence: str) -> bool:
    s = sentence.strip()

   # Very short sentences are often fragments; very long ones are typically lists or noisy text
    if len(s) < 40 or len(s) > 500:
        return False

    # Sentences with too few alphabetic characters are likely schedules, addresses, or tables
    if sum(c.isalpha() for c in s) / len(s) < 0.55:
        return False

    # Filter out contact information that may have survived preprocessing
    if RE_EMAIL.search(s) or RE_PHONE.search(s):
        return False

    # Filter train schedules and similar structured records
    if re.search(r"№\d+|–{2,}|—{2,}|\b\d{1,2}:\d{2}\b", s):
        return False

    # Filter navigation-style lists separated by "-"
    if s.count("—") >= 2:
        return False

    return True


def load_sentences_from_folder(folder_path: str) -> list[str]:
    sentences = []

    # Extract informative sentences from every file in the folders
    for root, _, files in os.walk(folder_path):
        for file in files:

            with open(os.path.join(root, file), "r", encoding="utf-8") as f:
                raw = f.read()

            for sent in split_sentences(clean_text(raw)):
                if is_informative(sent):
                    sentences.append(sent)

    return sentences

In [ ]:
eu_sentences = load_sentences_from_folder("/kaggle/input/datasets/alessandroisceri/text-corpus/eu")

X, y = preprocess(eu_sentences)
print(X)
print(y)

# All arrays share the same first dimension (number of samples)
n = len(eu_sentences) - 1 # Last sentence has no consecutive -> exclude it

# Shuffled indices
indices = np.random.permutation(n)

# Shuffle dicts
X_shuffled = {k: tf.gather(v, indices) for k, v in X.items()}
y_shuffled = {k: tf.gather(v, indices) for k, v in y.items()}

# split point (80% train, 20% val)
split = int(0.8 * n)

X_train = {k: v[:split] for k, v in X_shuffled.items()}
X_val   = {k: v[split:] for k, v in X_shuffled.items()}

y_train = {k: v[:split] for k, v in y_shuffled.items()}
y_val   = {k: v[split:] for k, v in y_shuffled.items()}

print("generated datasets")

{'token_ids': <tf.Tensor: shape=(45931, 512), dtype=int32, numpy=
array([[  101,  1998, 24285, ...,     0,     0,     0],
       [  101,   103,  2003, ...,     0,     0,     0],
       [  101,  2130,  2065, ...,     0,     0,     0],
       ...,
       [  101,  4499,  2013, ...,     0,     0,     0],
       [  101, 11199,  3334, ...,     0,     0,     0],
       [  101,  2036,  2031, ...,     0,     0,     0]], dtype=int32)>, 'padding_mask': <tf.Tensor: shape=(45931, 512), dtype=bool, numpy=
array([[ True,  True,  True, ..., False, False, False],
       [ True,  True,  True, ..., False, False, False],
       [ True,  True,  True, ..., False, False, False],
       ...,
       [ True,  True,  True, ..., False, False, False],
       [ True,  True,  True, ..., False, False, False],
       [ True,  True,  True, ..., False, False, False]])>, 'segment_ids': <tf.Tensor: shape=(45931, 512), dtype=int32, numpy=
array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ...,

In [ ]:
def make_dataset(X, y, batch_size, shuffle=False):
    # Build a TensorFlow dataset from input features and ground-truth labels
    dataset = tf.data.Dataset.from_tensor_slices((
        {
            "token_ids": X["token_ids"],
            "padding_mask": X["padding_mask"],
            "segment_ids": X["segment_ids"]
        },
        {
            "cls_output": y["cls_output"],
            "token_pred_output": y["token_pred_output"]
        }
    ))

    if shuffle:
        # Shuffle training examples at each epoch
        dataset = dataset.shuffle(buffer_size=len(X["token_ids"]), reshuffle_each_iteration=True)

    # Batch and prefetch examples for efficient input pipelining
    return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

# Serialize the pretraining dataset to TFRecord format
train_dataset = make_dataset(X_train, y_train, batch_size=PRETRAINING_BATCH_SIZE, shuffle=True)
val_dataset   = make_dataset(X_val,   y_val,   batch_size=PRETRAINING_BATCH_SIZE)

# Convert scalar or array values into an Int64List feature
def _int64_feature(values):
    if isinstance(values, (list, tuple, np.ndarray)):
        return tf.train.Feature(int64_list=tf.train.Int64List(value=values))
    return tf.train.Feature(int64_list=tf.train.Int64List(value=[values]))

# Create a TFRecord example containing model inputs and targets
def create_example(x, y):
    feature = {
        "token_ids": _int64_feature(x["token_ids"]),
        "padding_mask": _int64_feature(x["padding_mask"].astype("int64")),
        "segment_ids": _int64_feature(x["segment_ids"]),
        "cls_output": _int64_feature(y["cls_output"]),
        "token_pred_output": _int64_feature(y["token_pred_output"]),
    }

    return tf.train.Example(features=tf.train.Features(feature=feature))

# Write the dataset to disk as a TFRecord file
def write_tfrecord(X, y, filename):
    
    # Number of sentence pairs to serialize
    n = X["token_ids"].shape[0]

    with tf.io.TFRecordWriter(filename) as writer:
        for i in range(n):
            # Serialize a single training example
            example = create_example(
                {
                    "token_ids": X["token_ids"][i].numpy(),
                    "padding_mask": X["padding_mask"][i].numpy(),
                    "segment_ids": X["segment_ids"][i].numpy(),
                },
                {
                    "cls_output": y["cls_output"][i].numpy(),
                    "token_pred_output": y["token_pred_output"][i].numpy(),
                },
            )
            writer.write(example.SerializeToString())

# Persist training and validation datasets for later reuse
write_tfrecord(X_train, y_train, "/kaggle/working/train.tfrecord")
write_tfrecord(X_val, y_val, "/kaggle/working/val.tfrecord")